# Tutorial 1: Clifford Neural Score Matching (`Cl(2,0)`), Fokker–Planck Stationarity & Multi-Well Ablations

This tutorial accompanies the ML4PS @ NeurIPS 2026 paper:
> **"A-Block: Active Form Dynamics via Clifford Neural Score Matching"** — *ML4PS @ NeurIPS 2026*

### What You Will Learn
1. **Multi-Well Conformational Landscape:** Constructing the 3-basin free energy surface $V(\mathbf{x})$ (Basins A, B, C) at $k_B T = 0.6\text{ kcal/mol}$.
2. **Denoising Score Matching (DSM) & Energy Scale Calibration:** Training `CliffordScoreNet` without ground-truth force labels and verifying why DSM learns the dimensionless potential $V_\theta(\mathbf{x})$ scaled by $\alpha_E \approx 1.678\text{ (kcal/mol)}^{-1}$.
3. **Fokker\u2013Planck Stationarity vs. Active Confinement (Proposition 1):** Understanding why the Clifford rotor current $J_{\text{irr}}$ is divergence-free ($\nabla \cdot J_{\text{irr}} = 0$) at the Boltzmann density $\rho_{\text{B}}$.
4. **Component Ablations (Table 1 & Figure 1):** Comparing Brownian Dynamics, Standard Langevin ($\kappa=1, \sigma=0$), **Dilator-Only** ($\sigma=0, \kappa$), Rotor-Only ($\kappa=0, \sigma$), and **A-Block Full Clifford** ($(\sigma I - \kappa)\nabla V_\theta$).

In [ ]:
import os, sys, json
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure repo root is on sys.path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.potentials import MolecularMultiWellPotential, true_score
from src.models import CliffordScoreNet
from src.training import train_score_model
from src.samplers import run_multiwell_benchmarks
from src.metrics import get_boltzmann_reference, calc_w1, calc_fes_rmse, calc_basin_coverage

torch.manual_seed(42)
np.random.seed(42)
potential_fn = MolecularMultiWellPotential()
print('Initialized 3-Basin Conformational Potential:', potential_fn.centers.tolist())

## 1. Train or Load `CliffordScoreNet` and Inspect Energy Scale Calibration ($\alpha_E$)

Because DSM regresses $-\nabla V_\theta(\tilde{\mathbf{x}})$ against $-\boldsymbol{\epsilon}/\sigma_n^2$ (which targets $-\nabla V(\mathbf{x}) / (k_B T)$), the learned potential $V_\theta(\mathbf{x})$ naturally scales by $\alpha_E \approx 1.67\text{ (kcal/mol)}^{-1}$.

In [ ]:
ckpt_path = os.path.join(repo_root, 'output', 'clifford_score_net.pt')
if os.path.exists(ckpt_path):
    model = CliffordScoreNet(hidden_dim=64)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=True))
    print(f'Loaded pre-trained checkpoint from {ckpt_path}')
else:
    model = train_score_model(potential_fn, true_score, num_samples=12000, epochs=250)

with torch.no_grad():
    c_pts = potential_fn.centers
    v_true = potential_fn(c_pts)
    v_pred, sig_c, kap_c = model(c_pts)
    print('True Basin Energies V(c) [kcal/mol]:', v_true.numpy().round(3))
    print('Learned Dimensionless V_theta(c):   ', v_pred.squeeze().numpy().round(3))
    print('Basin Dilator kappa(c):             ', kap_c.squeeze().numpy().round(3))
    print('Basin Rotor chirality sigma(c):     ', sig_c.squeeze().numpy().round(3))

## 2. Run the 5-Method Multi-Well Benchmark (Including `Dilator-Only` Ablation)

Here we initialize $N = 2{,}000$ particles inside **Basin A** $(-1.0, 1.0)$ and integrate for $T = 1{,}000$ steps ($\Delta t = 0.015$). Notice how:
- **Dilator-Only** ($\sigma=0, \kappa$) calibrates the effective temperature inside the wells ($\text{FES RMSE} = 1.064\text{ kcal/mol}$ vs. $1.476$ for $\kappa=1$), but suffers a 35.4% drop in inter-basin transition rate ($13.86$ transitions/particle).
- **A-Block Full Clifford** ($(\sigma I - \kappa)\nabla V_\theta$) achieves the best FES RMSE (**1.059 kcal/mol**) while boosting inter-basin transitions by **+62.2%** (**22.48** transitions/particle).

In [ ]:
samplers, trajectories, times, extra_stats = run_multiwell_benchmarks(
    model, num_agents=2000, num_steps=1000, dt=0.015, seed=42, include_dilator_ablation=True
)
gt_particles = get_boltzmann_reference(potential_fn, num_agents=2000, kBT=0.6)

print(f"{'Method':<34} | {'FES RMSE':<10} | {'W1 Dist':<8} | {'Term B/C':<9} | {'Visit B/C':<9} | {'Trans/Pt'}")
print('-' * 90)
for name in samplers:
    pts_np = samplers[name].detach().numpy()
    rmse = calc_fes_rmse(pts_np, potential_fn, kBT=0.6, bins=35)
    w1 = calc_w1(pts_np, gt_particles)
    cov = calc_basin_coverage(samplers[name])
    vis = extra_stats[name]['visited_bc_pct']
    tr = extra_stats[name]['transitions_per_particle']
    print(f"{name:<34} | {rmse:<10.3f} | {w1:<8.3f} | {cov:<8.1f}% | {vis:<8.1f}% | {tr:.2f}")